# Step 07.4 — Position Information：Transformer 怎样知道“第几个 token”？

7.3 已经得到：

```text
token IDs (B,T)
   ↓ token embedding
hidden vectors (B,T,C)
```

但 token embedding 只回答：**这个 token 是谁？**

它并没有显式回答：**这个 token 出现在第几个位置？**

这一节完整解决 5 个问题：

1. 为什么 token identity 与 position 是两种不同信息；
2. RNN 为什么天然顺序敏感，而 pure self-attention 为什么具有 permutation equivariance；
3. learned absolute positional embedding 怎样工作；
4. causal mask 与 positional encoding 的关系，以及 learned absolute position 的局限；
5. 怎样可视化 position table，并为下一节 token + position addition 做准备。

## 1. Token identity 不等于 Position identity

假设 token `A` 的 ID 是 2。

无论它出现在 position 0 还是 position 5：

```text
token_embedding(A) = wte.weight[2]
```

都是同一行 parameter。

例如：

```text
sequence 1: A B C
sequence 2: C B A
```

token embedding lookup 只是把同样三行向量换了顺序：

```text
A B C -> [e_A, e_B, e_C]
C B A -> [e_C, e_B, e_A]
```

`e_A` 本身不会因为从第 0 位移动到第 2 位而改变。

In [ ]:
import mlx.core as mx
import mlx.nn as nn

V = 4
C = 6
wte = nn.Embedding(V, C)

seq_abc = mx.array([0, 1, 2])
seq_cba = mx.array([2, 1, 0])

emb_abc = wte(seq_abc)
emb_cba = wte(seq_cba)
mx.eval(emb_abc, emb_cba)

print("A at position 0 in ABC equals A at position 2 in CBA:")
print(bool(mx.allclose(emb_abc[0], emb_cba[2])))

print("B at position 1 in both sequences:")
print(bool(mx.allclose(emb_abc[1], emb_cba[1])))

所以当前表示拥有：

```text
token identity
```

但没有显式注入：

```text
absolute position identity
```

注意：数组本身当然有第 0、1、2 行；问题不是“电脑不知道数组顺序”，而是**模型的向量内容里还没有明确的位置特征**。

## 2. 为什么 RNN 天然顺序敏感，而 Pure Self-Attention 具有 Permutation Equivariance？

先看 RNN 的最小递推思想：

```text
h_t = f(x_t, h_{t-1})
```

第 t 步依赖第 t-1 步，所以 token 被送入的先后顺序会改变整条计算路径。

我们甚至不需要完整 RNN，就用一个玩具递推：

```text
h_t = x_t + 0.5 * h_(t-1)
```

In [ ]:
def toy_recurrence(sequence):
    h = 0.0
    states = []
    for x in sequence:
        h = x + 0.5 * h
        states.append(h)
    return states

forward_order = [1.0, 2.0, 3.0]
reverse_order = [3.0, 2.0, 1.0]

print("1,2,3 ->", toy_recurrence(forward_order))
print("3,2,1 ->", toy_recurrence(reverse_order))

顺序一换，所有后续 hidden states 都可能变，因为 recurrence 强制了一条时间链。

Transformer 的 pure self-attention 没有这条 `t-1 -> t` 的递推链。

这里引入一个新术语：**permutation equivariance（置换等变）**。

它不是 permutation invariance。

```text
invariant:
输入重新排列 -> 输出完全不变

equivariant:
输入重新排列 -> 输出跟着以同样方式重新排列
```

对不带位置编码、也不带 mask 的 pure self-attention，如果把 token rows 用某个 permutation P 重排，那么输出 rows 也会按 P 重排。

### 为什么会这样？先看一个极简数学预览

以后正式学 Attention 时我们会有：

```text
Q = X W_Q
K = X W_K
V = X W_V
```

如果 P 只是“重排行”的操作，那么：

```text
X' = P X
Q' = P Q
K' = P K
V' = P V
```

pairwise score：

```text
Q' K'^T
= P Q K^T P^T
```

所以所有 token-token 关系只是同步换了行和列；最后输出也同步换行。

这就是 pure self-attention 本身不会凭空制造“这是第 0 位、第 1 位”的绝对位置身份。

## 3. Learned Absolute Positional Embedding：给每个位置一行可学习向量

我们的教学 MiniGPT 使用最直观的方法：再建一张 embedding table，但这次不是按 token ID 查，而是按 position ID 查。

```python
wpe = nn.Embedding(block_size, C)
```

其中：

```text
wpe.weight.shape = (block_size, C)
```

每一行含义：

```text
row 0 -> position 0 的 C 维向量
row 1 -> position 1 的 C 维向量
row 2 -> position 2 的 C 维向量
...
```

所以 token table `wte` 回答“是谁”，position table `wpe` 回答“在哪”。

In [ ]:
block_size = 8
C = 6

wpe = nn.Embedding(block_size, C)

T = 4
positions = mx.arange(T)
pos_emb = wpe(positions)

mx.eval(pos_emb)

print("positions       :", positions)
print("positions shape :", positions.shape)
print("wpe.weight shape:", wpe.weight.shape)
print("pos_emb shape   :", pos_emb.shape)

这里：

```text
positions.shape = (T,)
wpe(positions)  = (T,C)
```

它没有 Batch axis，因为同一个 batch 中通常所有 sequence 都使用相同的相对位置编号：

```text
0,1,2,...,T-1
```

下一节会利用 broadcasting，把同一个 `(T,C)` position table 加到每个 batch sample 的 `(T,C)` token embeddings 上。

### 参数量

learned positional embedding 的参数量是：

```text
block_size * C
```

例如我们的教学配置若：

```text
block_size = 64
C = 128
```

那么 position embedding 参数量：

```text
64 * 128 = 8192
```

## 4. Causal Mask 与 Positional Encoding 不是同一件事

这里必须做一个精确区分。

前面说 pure self-attention permutation-equivariant，指的是：

```text
没有 positional signal
没有 causal mask
```

的普通 self-attention。

但 GPT 后面还会加入 triangular causal mask：

```text
position t 只能看 positions <= t
```

这个 mask 本身依赖 position index，所以它会打破任意 permutation 下的完全对称，并提供一部分顺序结构。

因此不能简单说：

> “只要没有 position embedding，causal Transformer 就完全不知道任何顺序。”

更准确的是：

> causal mask 定义**可见性顺序**；positional representation 显式提供**位置身份/位置特征**。

### Learned Absolute Position 的优点与局限

优点：

- 非常简单；
- 和 token embedding 一样是普通可训练 parameter；
- 非常适合从零理解 GPT。

局限：

- 每个 absolute position 都有独立 row，结构本身不直接表达“position 10 比 position 9 远 1”；
- 训练只覆盖 `0 ... block_size-1`；
- 如果没有额外方法，不能自然查询 `position >= block_size`；
- 对超出训练长度的 extrapolation 不友好。

历史上，原始 Transformer 使用 sinusoidal positional encoding；早期 GPT 系列常使用 learned absolute positional embeddings。现代 LLM 中 RoPE 很常见，我们到 Step 22 再系统讲。

## 5. 可视化 Position Table，并看见“同 token，不同位置”还缺的最后一步

`wpe.weight` 本身也是一个二维 parameter matrix：

```text
(block_size, C)
```

所以可以像 token embedding 一样用 heatmap 看。

这里只是在观察参数，不要给每一列强行赋予人类意义。

In [ ]:
import matplotlib.pyplot as plt

mx.eval(wpe.weight)
pos_table = wpe.weight.tolist()

fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(pos_table, aspect="auto")
ax.set_xlabel("feature dimension c")
ax.set_ylabel("absolute position t")
ax.set_title("Learned absolute positional embedding table")
fig.colorbar(image, ax=ax, label="parameter value")
plt.show()

现在构造一个重复 token：

```text
A A A
```

token embedding：

```text
e_A, e_A, e_A
```

完全一样。

position embedding：

```text
p_0, p_1, p_2
```

不同。

因此下一节我们会得到：

```text
position 0: e_A + p_0
position 1: e_A + p_1
position 2: e_A + p_2
```

同一个 token 终于会因为所处位置不同而拥有不同的初始 hidden representation。

In [ ]:
wte_small = nn.Embedding(3, C)
same_tokens = mx.array([1, 1, 1])
positions = mx.arange(3)

tok = wte_small(same_tokens)
pos = wpe(positions)

mx.eval(tok, pos)

print("token vectors equal at pos0/pos1:", bool(mx.allclose(tok[0], tok[1])))
print("position vectors equal at pos0/pos1:", bool(mx.allclose(pos[0], pos[1])))
print("token shape   :", tok.shape)
print("position shape:", pos.shape)

### 7.4 最终结论

- token embedding 编码 token identity，但同一 token 在不同位置读取同一 `wte` row；
- RNN 通过 `h_t=f(x_t,h_{t-1})` 的递推路径天然对输入顺序敏感；
- 不带 mask / positional signal 的 pure self-attention 对 token permutation 是 equivariant：输入重排，输出对应重排；
- learned absolute position embedding 使用 `wpe.weight.shape=(block_size,C)`，按 `0...T-1` 查出 `(T,C)`；
- causal mask 与 positional encoding 不同：mask 规定谁能看谁，position representation 提供位置身份/特征；
- learned absolute positions 简单但不天然表达相对距离，也不自然支持超出 `block_size` 的新位置；
- 下一节 7.5：真正执行 `(B,T,C) + (T,C)` broadcasting，解释为什么 GPT 选择“相加”而不是 concatenate，以及这样做保留了什么结构、牺牲了什么。